# Lesson 22: Deep Reinforcement Learning (`Gymnasium` + PyTorch Policy Gradients)

In this notebook, you will:
1. Train a PyTorch **REINFORCE Policy Gradient** agent on `CartPole-v1` using `gymnasium`.
2. Complete the **Lab**: Explore `MountainCar-v0` with reward shaping.

In [ ]:
!pip install -q gymnasium
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

env = gym.make("CartPole-v1")
policy = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 2))
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99

for ep in range(150):
    state, _ = env.reset(seed=ep)
    log_probs, rewards = [], []
    done = False
    while not done:
        probs = F.softmax(policy(torch.tensor(state, dtype=torch.float32)), dim=-1)
        dist = Categorical(probs)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        state, r, term, trunc, _ = env.step(action.item())
        rewards.append(r)
        done = term or trunc

    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    loss = -sum(lp * g for lp, g in zip(log_probs, returns))
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1:3d} | Total Reward: {sum(rewards):.0f}")

In [ ]:
# Lab: MountainCar-v0 environment check
mc = gym.make("MountainCar-v0")
obs, _ = mc.reset(seed=0)
print("MountainCar observation [position, velocity]:", obs)